# AI Diet Coach Agent — Week 2

Turns the Week 1 RAG pipeline into a tool-calling agent.

## Question 1 — Tool Design

| Tool | Description | Inputs | Returns | Priority |
|------|-------------|--------|---------|----------|
| `search_recipes` | Full-text search over recipes by goal, ingredient, cuisine, or mood | `query: str` | up to 5 recipe objects | **Essential** |
| `filter_by_max_cook_time` | Returns every recipe that fits within a time budget | `max_minutes: int` | list of recipe objects | **Essential** |
| `filter_by_category` | Returns every recipe in a food category (e.g. Chicken, Seafood, Vegetarian) | `category: str` | list of recipe objects | Essential |
| `get_recipe_details` | Retrieves full ingredients + instructions for one recipe by name | `name: str` | single recipe object or error | Essential |

**Rationale:**
- `search_recipes` is the core RAG tool inherited from Week 1.
- `filter_by_max_cook_time` lets the agent answer 'I only have 15 minutes' queries without guessing.
- `filter_by_category` covers dietary-restriction queries ('only chicken dishes').
- `get_recipe_details` lets the agent drill into a specific recipe when it needs full instructions.
- A `build_meal_plan` tool was considered but rejected — the agent's LLM reasoning handles planning once it has recipes, so a dedicated planning tool would just duplicate logic.

## Setup

In [1]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Recipe Data & Build Index

In [2]:
with open('../data/recipes.json') as f:
    documents = json.load(f)

print(f'Loaded {len(documents)} recipes')

categories_available = sorted({d['category'] for d in documents})
print('Categories:', categories_available)

Loaded 201 recipes
Categories: ['Beef', 'Breakfast', 'Chicken', 'Dessert', 'Goat', 'Lamb', 'Miscellaneous', 'Pasta', 'Pork', 'Seafood', 'Side', 'Starter', 'Vegan', 'Vegetarian']


In [3]:
index = Index(
    text_fields=["name", "category", "area", "ingredients", "instructions"],
    keyword_fields=["category", "area"],
)
index.fit(documents)
print('Index built on', len(documents), 'recipes')

Index built on 201 recipes


## 2. Define Tool Functions

In [4]:
def search_recipes(query: str) -> list:
    return index.search(query, num_results=5)


def filter_by_max_cook_time(max_minutes: int) -> list:
    results = [
        doc for doc in documents
        if doc.get('cooking_time_minutes', 9999) <= max_minutes
    ]
    return results[:10]


def filter_by_category(category: str) -> list:
    results = [
        doc for doc in documents
        if doc.get('category', '').lower() == category.lower()
    ]
    return results[:10]


def get_recipe_details(name: str) -> dict:
    name_lower = name.lower()
    for doc in documents:
        if name_lower in doc['name'].lower():
            return doc
    return {"error": f"No recipe found matching '{name}'"}

## 3. Tool Schemas (OpenAI function-calling format)

In [5]:
search_recipes_tool = {
    "type": "function",
    "name": "search_recipes",
    "description": (
        "Search the recipe database for meals that match a query. "
        "Use this when the user describes a goal, ingredient, cuisine type, "
        "or any free-text request like 'high protein' or 'low calorie pasta'."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Natural-language search query, e.g. 'high protein chicken'"
            }
        },
        "required": ["query"]
    }
}

filter_by_max_cook_time_tool = {
    "type": "function",
    "name": "filter_by_max_cook_time",
    "description": (
        "Return recipes that can be cooked within a given number of minutes. "
        "Use this when the user says they have limited time."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "max_minutes": {
                "type": "integer",
                "description": "Maximum cooking time in minutes"
            }
        },
        "required": ["max_minutes"]
    }
}

filter_by_category_tool = {
    "type": "function",
    "name": "filter_by_category",
    "description": (
        "Return all recipes in a specific food category such as 'Chicken', "
        "'Beef', 'Seafood', 'Vegetarian', 'Pasta', etc. "
        "Use this when the user specifies a dietary preference or protein type."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "category": {
                "type": "string",
                "description": "Food category, e.g. 'Chicken', 'Seafood', 'Vegetarian'"
            }
        },
        "required": ["category"]
    }
}

get_recipe_details_tool = {
    "type": "function",
    "name": "get_recipe_details",
    "description": (
        "Retrieve the full ingredients list and step-by-step instructions for a "
        "specific recipe by name. Use this when the user asks 'how do I make X' "
        "or after finding a recipe the user wants to know more about."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "The recipe name or partial name to look up"
            }
        },
        "required": ["name"]
    }
}

tools = [
    search_recipes_tool,
    filter_by_max_cook_time_tool,
    filter_by_category_tool,
    get_recipe_details_tool,
]

## 4. Tool Dispatcher

In [6]:
TOOL_REGISTRY = {
    'search_recipes': search_recipes,
    'filter_by_max_cook_time': filter_by_max_cook_time,
    'filter_by_category': filter_by_category,
    'get_recipe_details': get_recipe_details,
}


def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    fn = TOOL_REGISTRY.get(name)
    if fn is None:
        result = {"error": f"Unknown tool '{name}'"}
    else:
        result = fn(**arguments)

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

## 5. Agent Instructions & Loop

In [7]:
instructions = """
You are an AI diet coach helping users reach their weight-loss goals through
personalized meal recommendations.

You have access to a database of 201 recipes. Use your tools to answer every
question — never invent recipes that are not in the database.

When to call each tool:
- search_recipes: when the user describes a goal, ingredient, cuisine, or mood
  (e.g. 'high protein', 'quick dinner', 'Asian food', 'something with chicken').
  Always start here for general queries.
- filter_by_max_cook_time: when the user mentions a time limit
  (e.g. 'I only have 15 minutes', 'quick', 'under 20 min').
  You may combine this with search_recipes to refine results.
- filter_by_category: when the user specifies a protein or food type
  (e.g. 'only chicken', 'vegetarian options', 'seafood dishes').
- get_recipe_details: when the user wants the full recipe for a specific dish
  (e.g. 'how do I make the stir-fry?', 'give me the full recipe').

How to respond:
- For each recommended recipe include: name, category, cooking time, key
  ingredients, and one sentence explaining why it fits the user's goal.
- If the user is on a weight-loss programme, prefer lower-calorie options and
  lean proteins. Mention calorie-related reasoning when relevant.
- If no suitable recipes are found, say so honestly and give a general
  dietary tip instead.
- Be concise and supportive — you are a coach, not a cookbook index.
""".strip()

In [8]:
def run_agent(user_question: str, model: str = 'gpt-4o-mini') -> str:
    message_history = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_question},
    ]

    iteration = 1
    final_answer = None

    while True:
        print(f"--- iteration {iteration} ---")

        response = openai_client.responses.create(
            model=model,
            input=message_history,
            tools=tools,
        )

        message_history.extend(response.output)
        has_tool_calls = False

        for item in response.output:
            if item.type == 'function_call':
                print(f"  TOOL CALL: {item.name}({item.arguments})")
                tool_result = make_call(item)
                message_history.append(tool_result)
                has_tool_calls = True

            elif item.type == 'message':
                final_answer = item.content[0].text

        iteration += 1

        if not has_tool_calls:
            break

    print()
    return final_answer

## 6. Test the Agent (Question 3)

Three queries covering different tool paths:
1. General search → `search_recipes`
2. Time-constrained search → `filter_by_max_cook_time` + `filter_by_category`
3. Edge case: out-of-scope query

### Query 1 — General goal query (triggers `search_recipes`)

In [9]:
q1 = "I want to lose weight. Can you suggest a high-protein dinner that isn't boring?"
print("USER:", q1)
print()
answer1 = run_agent(q1)
print("AGENT:", answer1)

USER: I want to lose weight. Can you suggest a high-protein dinner that isn't boring?

--- iteration 1 ---


  TOOL CALL: search_recipes({"query":"high protein dinner"})
--- iteration 2 ---



AGENT: Here are some high-protein dinner options that aren't boring and will help you on your weight-loss journey:

1. **Beef and Broccoli Stir-Fry**
   - **Category:** Beef
   - **Cooking Time:** 14 minutes
   - **Key Ingredients:** Sirloin steak, broccoli, garlic, oyster sauce
   - **Why It Fits:** This dish is packed with protein from the sirloin and is quick to make, making it both satisfying and healthy.

2. **Salmon Noodle Wraps**
   - **Category:** Seafood
   - **Cooking Time:** 22 minutes
   - **Key Ingredients:** Salmon, rice noodles, spring onions, lime
   - **Why It Fits:** With rich omega-3 fatty acids and high protein content, this dish is flavorful and perfect for dinner.

3. **Stir-Fried Chicken with Chillies & Basil**
   - **Category:** Chicken
   - **Cooking Time:** 23 minutes
   - **Key Ingredients:** Chicken, garlic, red chili, basil
   - **Why It Fits:** A deliciously spicy option, this stir-fry is not only high in protein but also bursting with vibrant flavors.

4

### Query 2 — Time-constrained + category (triggers `filter_by_max_cook_time` and `filter_by_category`)

In [10]:
q2 = "I only have 15 minutes tonight. What vegetarian meals can I make?"
print("USER:", q2)
print()
answer2 = run_agent(q2)
print("AGENT:", answer2)

USER: I only have 15 minutes tonight. What vegetarian meals can I make?

--- iteration 1 ---


  TOOL CALL: filter_by_max_cook_time({"max_minutes":15})
--- iteration 2 ---


  TOOL CALL: filter_by_category({"category":"Vegetarian"})
--- iteration 3 ---



AGENT: Here are some vegetarian meals you can prepare in 15 minutes or less:

### 1. **Beetroot Latkes**
- **Category:** Vegetarian
- **Cooking Time:** 15 minutes
- **Key Ingredients:** Beetroot, Egg, Plain Flour, Greek Yogurt, Mint
- **Why It Fits:** These flavorful latkes are quick to prepare and perfect for a light dinner. They include nutritious beetroot, which is low in calories and high in vitamins.

### 2. **Chinese Tomato Egg Stir Fry**
- **Category:** Vegetarian
- **Cooking Time:** 8 minutes
- **Key Ingredients:** Plum Tomatoes, Eggs, Sesame Seed Oil, Spring Onions
- **Why It Fits:** This dish is quick and highlights simple, fresh ingredients. The tomatoes and eggs provide protein while keeping the meal light and healthy.

### 3. **Egg Drop Soup**
- **Category:** Vegetarian
- **Cooking Time:** 35 minutes (too long for your time constraint; included for reference)
- **Key Ingredients:** Chicken Stock, Peas, Mushrooms
- **Why It Fits:** Although it takes longer to make, if you 

### Query 3 — Edge case: out-of-scope request

In [11]:
q3 = "Can you book me a table at the nearest Japanese restaurant and order for me?"
print("USER:", q3)
print()
answer3 = run_agent(q3)
print("AGENT:", answer3)

USER: Can you book me a table at the nearest Japanese restaurant and order for me?

--- iteration 1 ---



AGENT: I can't make reservations or place orders, but I can help you find some great Japanese recipes or recommend dishes if you're planning to cook at home! Would you like some recipe ideas?


### Bonus — Full recipe drill-down (triggers `search_recipes` then `get_recipe_details`)

In [12]:
q4 = "Find me a quick Asian stir-fry recipe and give me the full cooking instructions."
print("USER:", q4)
print()
answer4 = run_agent(q4)
print("AGENT:", answer4)

USER: Find me a quick Asian stir-fry recipe and give me the full cooking instructions.

--- iteration 1 ---


  TOOL CALL: search_recipes({"query":"quick Asian stir-fry"})
--- iteration 2 ---


  TOOL CALL: get_recipe_details({"name":"Prawn stir-fry"})
--- iteration 3 ---



AGENT: ### Prawn Stir-Fry

- **Category**: Seafood
- **Cooking Time**: 4 minutes
- **Key Ingredients**: Raw tiger prawns, green chilli, garlic, coriander, lime juice, fish sauce, groundnut oil, ginger, spring onions, red pepper, water chestnuts, bean sprouts, soy sauce, rice noodles.

#### Cooking Instructions:

1. **Prepare the Prawns**: In a bowl, combine the prawns with a chopped green chilli, crushed garlic, snipped coriander stalks, and caster sugar. Blend together with half of the lime juice and fish sauce, then pour this mixture over the prawns.

2. **Stir-Fry the Veggies**: Heat 1 tablespoon of oil in a wok. Add ginger and spring onions, frying for 1 minute. Then add the sliced red pepper and fry for another minute until softened. Toss in the water chestnuts and bean sprouts until the sprouts begin to wilt. Add soy sauce and a good grind of black pepper, then transfer everything to a serving dish.

3. **Cook the Prawns**: In the same wok, add the remaining oil. Add the prawns 